# Imports

In [3]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns	
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, auc, roc_auc_score, roc_curve, make_scorer, precision_recall_curve, roc_auc_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

# Tabela

In [4]:
df = pd.read_csv('synthetic_coffee_health_10000(in).csv')
df

,ID,Age,Gender,Country,Coffee_Intake,Caffeine_mg,Sleep_Hours,Sleep_Quality,BMI,Heart_Rate,Stress_Level,Physical_Activity_Hours,Health_Issues,Occupation,Smoking,Alcohol_Consumption
0,1,40,Male,Germany,3.5,328.1,7.5,Good,24.9,78,Low,14.5,NaN,Other,0,0
1,2,33,Male,Germany,1.0,94.1,6.2,Good,20.0,67,Low,11.0,NaN,Service,0,0
2,3,42,Male,Brazil,5.3,503.7,5.9,Fair,22.7,59,Medium,11.2,Mild,Office,0,0
3,4,53,Male,Germany,2.6,249.2,7.3,Good,24.7,71,Low,6.6,Mild,Other,0,0
4,5,32,Female,Spain,3.1,298.0,5.3,Fair,24.1,76,Medium,8.5,Mild,Student,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,50,Female,Japan,2.1,199.8,6.0,Fair,30.5,50,Medium,10.1,Moderate,Healthcare,0,1
9996,9997,18,Female,UK,3.4,319.2,5.8,Fair,19.1,71,Medium,11.6,Mild,Service,0,0
9997,9998,26,Male,China,1.6,153.4,7.1,Good,25.1,66,Low,13.7,NaN,Student,1,1
9998,9999,40,Female,Finland,3.4,327.1,7.0,Good,19.3,80,Low,0.1,NaN,Student,0,0


## Metadados

In [8]:
def get_metadata(df, alpha=0.05, min_samples=20):
    colunas = df.columns
    tipos = [df[col].dtype.name for col in colunas]   # cleaner dtype names
    
    # Compute null counts once
    null_counts = df.isnull().sum()
    total_rows = df.shape[0]
    pct_nulls = (null_counts / total_rows * 100).round(2)
    
    cardinalidade = df.nunique()
    
    def test_normality(coluna):
        dtype = df[coluna].dtype
        if not pd.api.types.is_numeric_dtype(dtype):
            return None
        values = df[coluna].dropna()
        if len(values) < min_samples:
            return None
        stat, p = stats.normaltest(values)
        return p > alpha   # True if we cannot reject normality
    
    fl_normal = [test_normality(col) for col in colunas]
    
    metadata = pd.DataFrame({
        'Nome_Variavel': colunas,
        'Tipo_Dado': tipos,
        'Qtd_Nulos': null_counts.values,
        'Pct_Nulos': pct_nulls.values,
        'Cardinalidade': cardinalidade.values,
        'Normal': fl_normal
    })
    
    metadata = metadata.sort_values(by='Pct_Nulos', ascending=False).reset_index(drop=True)
    return metadata

# Usage
metadados_01 = get_metadata(df)
metadados_01

,Nome_Variavel,Tipo_Dado,Qtd_Nulos,Pct_Nulos,Cardinalidade,Normal
0,Health_Issues,object,5941,59.41,3,None
1,ID,int64,0,0.00,10000,False
2,Age,int64,0,0.00,59,False
3,Gender,object,0,0.00,3,None
4,Country,object,0,0.00,20,None
5,Coffee_Intake,float64,0,0.00,78,False
6,Caffeine_mg,float64,0,0.00,4277,False
7,Sleep_Hours,float64,0,0.00,71,False
8,Sleep_Quality,object,0,0.00,4,None
9,BMI,float64,0,0.00,220,False


# 1. EDA

## Variáveis Numéricas

In [9]:
num_cols = [col for col, dtype in df.dtypes if dtype in ['int64', 'float64'] and col not in ('ID')]

n = len(num_cols)
ncols = 3
nrows = -(-n//ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(15, 5*nrows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df.boxplot(column=col, ax=axes[i])
    axes[i].set_title(f'Boxplot of {col}', fontsize=12, fontweight='bold')
    axes[i].tick_params(axis='x', labelbottom=False)

for j in range(i+1, len(axes)):
    axes[j].axis('off')
    
plt.suptitle('Boxplots of Numeric Variables', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

TypeError: cannot unpack non-iterable numpy.dtypes.Int64DType object

Análise Exploratória de Dados (EDA)

A empresa Health&Life Analytics precisa entender melhor o perfil dos clientes e como o consumo de café se relaciona com aspectos de saúde.
Nosso primeiro passo será realizar uma análise exploratória completa, identificando padrões e potenciais problemas nos dados.
A Atividade

    Carregue o dataset em um notebook.
        Verifique dimensões, colunas, tipos de dados, nulos e duplicados.
        Gere estatísticas descritivas iniciais.

    Explore as variáveis numéricas.
        Crie histogramas, boxplots e distribuições para analisar tendências.

    Explore as variáveis categóricas.
        Crie gráficos de barras e pizzas para visualizar frequências.

    Investigue possíveis correlações.
        Exemplo: relação entre consumo de café e horas de sono, estresse e qualidade do sono.

    Documente seus achados no notebook com texto e gráficos explicativos.


Visualização e Insights

Após entender a estrutura dos dados, precisamos gerar visualizações e análises comparativas que tragam valor para o negócio.
O objetivo é comunicar de forma clara como o café, o estresse e outros fatores impactam o sono.
A Atividade

    Construa gráficos comparativos que relacionem variáveis do conjunto de dados.
        Utilize diferentes tipos de visualização (dispersão, boxplots, mapas de calor, entre outros).

    Faça análises segmentadas por grupos:
        Diferenças de sono por gênero.
        Idade e estresse em conjunto com consumo de café.

    Identifique padrões e escreva insights de negócio:
        “Clientes com alto consumo de cafeína dormem em média X horas a menos.”
        “O nível de estresse X altera [...] a qualidade do sono.”

    Prepare uma breve seção no notebook chamada Principais Descobertas, onde você lista os achados mais relevantes com gráficos.


Modelo preditivo

O time deseja prever a qualidade do sono (Sleep_Quality) com base nos hábitos e características dos clientes.
Nosso último passo será preparar os dados e implementar modelos de classificação.
A atividade

    Faça o pré-processamento dos dados:
        Remova ou trate colunas irrelevantes.
        Aplique codificação em variáveis categóricas.
        Crie pelo menos uma feature derivada.

    Divida os dados em treino e teste.

    Treine pelo menos dois modelos de classificação sendo, Sleep_Quality, a variável alvo (target).
        Compare resultados usando acurácia, matriz de confusão e/ou relatório de classificação.

    Documente a avaliação:
        Qual modelo performou melhor?
        Há overfitting ou underfitting?

    Salve o dataset final processado em CSV.

    Salve o modelo que se saiu melhor.

    Adicione uma seção Recomendações para o Negócio, explicando como os resultados podem ajudar a empresa a orientar clientes sobre hábitos de café e sono.


Durante nossa revisão, usamos como guia alguns pontos importantes que sua implementação deve seguir.

Recomendamos que você autoavalie seu projeto antes da submissão e observe se ele segue a maior parte desses pontos.
Pontos gerais

    O repositório Git submetido é acessível publicamente?
    O repositório contém um README.md explicando o objetivo do projeto, como rodar e quais bibliotecas são necessárias?
    O projeto roda corretamente em Jupyter Notebook ou Google Colab (sem erros de execução nas células)?
    O dataset está disponível ou indicado no repositório?

Entrega #1 – Análise exploratória (EDA)

    O notebook apresenta uma análise exploratória inicial?
    Foram gerados gráficos adequados para as variáveis numéricas?
    Foram gerados gráficos adequados para as variáveis categóricas?
    Há comentários claros e insights sobre os padrões encontrados?

Entrega #2 – Visualização e insights

    Foram criados gráficos comparativos relacionando consumo de café e sono?
    Houve segmentação por grupos?
    Há pelo menos uma seção de Principais Descobertas documentando os insights obtidos?
    Os gráficos estão legíveis (títulos, legendas, eixos claros)?

Entrega #3 – Modelo preditivo

    Foi feito pré-processamento dos dados (remoção de colunas irrelevantes, encoding de categóricas, criação de features)?
    Os dados foram corretamente divididos em treino e teste?
    Pelo menos dois modelos de classificação foram treinados e avaliados?
    Foram exibidas métricas de avaliação (acurácia, matriz de confusão, relatório de classificação)?
    O dataset final foi salvo em CSV?
    O modelo que teve a melhor performance foi salvo?
    Há uma seção Recomendações para o Negócio explicando como os resultados podem apoiar decisões?